# **PREPROCESAMIENTO DEL DATASET: GÉNEROS Y ARTISTAS**

In [75]:
import pandas as pd
import numpy as np
import re
import ast
import html
from pathlib import Path

In [76]:

# 1. Cargar el documento CSV
ruta_archivo = 'datasets scraping/dataset_fusionado.csv'

try:
    df = pd.read_csv(ruta_archivo)
    
    # 2. Obtener el número de filas y columnas usando .shape
    num_filas, num_columnas = df.shape
    print("=== INFORMACIÓN GENERAL ===")
    print(f"Número de filas: {num_filas}")
    print(f"Número de columnas: {num_columnas}\n")
    
    # 3. Mostrar los nombres de las columnas usando .columns
    print("=== NOMBRES DE LAS COLUMNAS ===")
    for columna in df.columns:
        print(f"- {columna}")
    print("\n")
    
    # 4. Ver cuántas filas están vacías (nulas) por columna usando .isnull().sum()
    print("=== VALORES VACÍOS POR COLUMNA ===")
    valores_vacios = df.isnull().sum()
    print(valores_vacios)

except FileNotFoundError:
    print(f"Error: No se encontró el archivo '{ruta_archivo}'. Verifica que el nombre esté escrito correctamente y esté en la misma carpeta que este script.")
except Exception as e:
    print(f"Ocurrió un error inesperado al leer el archivo: {e}")

=== INFORMACIÓN GENERAL ===
Número de filas: 10081
Número de columnas: 8

=== NOMBRES DE LAS COLUMNAS ===
- artist
- song
- lyrics
- genres
- genre_1
- genre_2
- genre_3
- <<<<<<< HEAD


=== VALORES VACÍOS POR COLUMNA ===
artist          3715
song            3715
lyrics          4814
genres          4449
genre_1         4449
genre_2         4466
genre_3         4467
<<<<<<< HEAD    9166
dtype: int64


In [77]:
# cargar dataset
DATA_PATH = "datasets scraping/dataset_fusionado.csv"

df = pd.read_csv(DATA_PATH)

print("Shape inicial:", df.shape)
display(df.head())
display(df.tail())

Shape inicial: (10081, 8)


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,<<<<<<< HEAD
0,/moonspell/,Heartshaped Abyss,Never resist\r\nThe firewalker\r\nHe has been ...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
1,/moonspell/,Let The Children Cum To Me...,"Hey you little Jesus' bride, why have you smil...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
2,/moonspell/,Memento Mori,"I am the moment, the soul\r\nThe moment that e...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
3,/moonspell/,Once It Was Ours!,Vanishing act inside the weak\r\nIn need of yo...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN
4,/moonspell/,Serpent Angel,Father Satan send the Serpent\r\nPoison me wit...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,NaN


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,<<<<<<< HEAD
10076,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10077,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10078,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [78]:
# Eliminar la columna HEAD
df = df.drop(columns=["<<<<<<< HEAD"], errors="ignore")

# Normalizar columnas
df.columns = df.columns.str.strip().str.lower()

expected_cols = ["artist", "song", "lyrics", "genres", "genre_1", "genre_2", "genre_3"]

missing_cols = [col for col in expected_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas en el dataset: {missing_cols}")

print("Columnas disponibles:")
print(df.columns.tolist())

Columnas disponibles:
['artist', 'song', 'lyrics', 'genres', 'genre_1', 'genre_2', 'genre_3']


# **PREPROCESADO DE ARTIST**

In [79]:

def clean_artist(text):
    """
    Limpieza ligera para nombres de artistas.
    Elimina barras (/), decodifica HTML y limpia espacios.
    """
    if pd.isna(text):
        return np.nan
    
    text = str(text)
    
    # Decodificar entidades HTML (ej: Beyonc&eacute; -> Beyoncé)
    text = html.unescape(text)
    
    # Eliminar URLs (por si el scraper capturó un enlace por error)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    
    # Normalizar comillas y apóstrofes
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )
    
    # Eliminar las barras "/" (reemplazándolas por un espacio para evitar fusionar palabras)
    text = text.replace("/", " ")
    
    # Normalizar saltos de línea y tabs
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    
    # Eliminar espacios múltiples y recortar los extremos
    text = re.sub(r"\s+", " ", text).strip()
    
    if text == "":
        return np.nan
    
    return text

# Aplicar la limpieza a la columna artist
df["artist_clean"] = df["artist"].apply(clean_artist)

# Mostrar una pequeña comprobación de los resultados
display(df[["artist", "artist_clean", "song"]].head(10))

,artist,artist_clean,song
0,/moonspell/,moonspell,Heartshaped Abyss
1,/moonspell/,moonspell,Let The Children Cum To Me...
2,/moonspell/,moonspell,Memento Mori
3,/moonspell/,moonspell,Once It Was Ours!
4,/moonspell/,moonspell,Serpent Angel
5,/moonspell/,moonspell,The Darkening
6,/moonspell/,moonspell,Upon The Blood Of Men
7,/running-wild/,running-wild,White Buffalo
8,/running-wild/,running-wild,Billy The Kid
9,/running-wild/,running-wild,Branded And Exiled


# **PREPROCESADO DE GENRES**

In [ ]:

# 1. Combinar las tres columnas de géneros en una sola serie (lista larga)
todos_los_generos = pd.concat([df['genre_1'], df['genre_2'], df['genre_3']])

# 2. Eliminar valores nulos (NaN) y obtener los valores únicos
generos_unicos = todos_los_generos.dropna().unique()

# 3. Limpiar espacios extra y poner en minúsculas para evitar duplicados (ej: "Pop" y " pop ")
# Convertimos a serie temporalmente para aplicar métodos de texto de pandas
generos_limpios = pd.Series(generos_unicos).str.strip().str.lower().unique()

# 4. (Opcional) Ordenar la lista alfabéticamente para que sea más fácil de leer
generos_limpios = sorted(generos_limpios)

print("=== GÉNEROS ÚNICOS EN EL DATASET ===")
print(f"Total de géneros distintos: {len(generos_limpios)}\n")
print(generos_limpios)

=== GÉNEROS ÚNICOS EN EL DATASET ===
Total de géneros distintos: 1258

['-death metal-', '00s', '1 star', '10 of 10 stars', '10 stars', '10s', '123 drippy sappy', '128 bpm', '1947', '1957', '1959', '1960s', '1962', '1965', '1967', '1969', '1970', '1971', '1973', '1976', '1977', '1978', '1979', '1980s', '1981', '1983', '1984', '1987', '1989', '1990', '1990s', '1991', '1992', '1993', '1994', '1996', '1997', '1998', '1999', '2 of 10 stars', '2000s', '2001', '2002', '2003', '2005', '2006', '2007', '2008', '2009', '2010', '2010s', '2012', '2013', '2013 single', '2014', '2015', '2015 gif', '2016', '2017', '2018 single', '2019', '2020', '2020s', '2021', '2022', '20th century', '2ne1', '3-star', '3l3ktr0 ch1x', '4 of 10 stars', '40', '40s', '5 of 10 stars', "50's", '50s', '50s rock n roll', '6 of 10 stars', "60's", '600th play', '60s', '7 of 10 stars', '70s', '70s blockbusters', '77davez-all-tracks', '8 of 10 stars', '80', '80 bpm', '80s', '80s hard rock', '80s power metal', '80s rhythm and bl

In [81]:

# ============================================================
# 1. Géneros válidos
# ============================================================

VALID_GENRES = {
    "acid house",
    "acid jazz",
    "acid rock",
    "acid techno",
    "afro-cuban jazz",
    "afro-pop",
    "afrobeats",
    "alt rock",
    "alt-country",
    "alternative",
    "alternative hip hop",
    "alternative metal",
    "alternative r&b",
    "alternative rock",
    "alternative/modern metal",
    "amapiano",
    "ambient",
    "american country",
    "anti-folk",
    "appalachian",
    "arabic pop",
    "art pop",
    "art rock",
    "avant-garde",
    "avant-garde jazz",
    "bachata",
    "baroque",
    "bass music",
    "bassline",
    "bebop",
    "bhangra",
    "big band",
    "black metal",
    "bluegrass",
    "blues",
    "blues rock",
    "bollywood",
    "boogie",
    "boom bap",
    "bossa nova",
    "britpop",
    "bro-country",
    "brostep",
    "bubblegum pop",
    "c-pop",
    "calypso",
    "canadian country",
    "carnatic classical",
    "celtic",
    "chamber pop",
    "chicago blues",
    "chicago drill",
    "chillout",
    "chillstep",
    "chiptune (8-bit)",
    "choral & liturgical",
    "classic blues",
    "classic country",
    "classic metal",
    "classic r&b",
    "classic rock",
    "classical",
    "cloud rap",
    "conscious hip hop",
    "contemporary country",
    "contemporary folk",
    "contemporary pop",
    "contemporary r&b",
    "contry",
    "cool jazz",
    "country",
    "country and western",
    "country blues",
    "country pop",
    "country rap",
    "country rock",
    "country two step",
    "countrypolitan",
    "cumbia",
    "cyberpunk",
    "dance",
    "dance-pop",
    "dancehall",
    "dangdut",
    "death metal",
    "deathcore",
    "deep house",
    "delta blues",
    "detroit techno",
    "disco",
    "dixieland",
    "doo-wop",
    "doom metal",
    "downtempo",
    "dream pop",
    "drill",
    "drum and bass",
    "dub",
    "dubstep",
    "early jazz",
    "early music",
    "east coast hip hop",
    "electro house",
    "electro pop",
    "electro swing",
    "electroclash",
    "electronic",
    "electronic & dance (edm)",
    "electropop",
    "emo",
    "emo rap",
    "emocore",
    "enka",
    "experimental",
    "expressionism",
    "extreme metal",
    "folk",
    "folk pop",
    "folk rock",
    "folkrock",
    "freak folk",
    "free jazz",
    "french house",
    "frenchcore",
    "funk",
    "fusion",
    "future bass",
    "g-funk",
    "gabber",
    "gagaku",
    "gamelan",
    "gangsta rap",
    "garage rock",
    "glam metal",
    "glitch",
    "go-go",
    "goa trance",
    "gospel",
    "gothic metal",
    "grand opera",
    "gregorian chant",
    "grime",
    "grunge",
    "gypsy jazz",
    "hard bop",
    "hard dance",
    "hard rock",
    "hard trance",
    "hardcore",
    "hardcore punk",
    "hardstyle",
    "heartland rock",
    "heavy metal",
    "highlife",
    "hindustani classical",
    "hip hop",
    "honky-tonk",
    "house",
    "hyperpop",
    "idm (intelligent dance music)",
    "illbient",
    "impressionism",
    "indie",
    "indie folk",
    "indie pop",
    "indie rock",
    "indietronica",
    "industrial techno",
    "italo disco",
    "j-pop",
    "jazz",
    "jazz funk",
    "jazz rap",
    "jazz-rock",
    "jump blues",
    "jungle",
    "jùjú",
    "k-hip hop",
    "k-pop",
    "klezmer",
    "krautrock",
    "kwaito",
    "latin hip hop",
    "latin jazz",
    "latin pop",
    "latin trap",
    "liquid drum and bass",
    "lo-fi hip hop",
    "mariachi",
    "mass",
    "math rock",
    "mbalax",
    "medieval",
    "melodic death metal",
    "memphis blues",
    "merengue",
    "metal",
    "metalcore",
    "midwest hip hop",
    "minimal techno",
    "minimalism",
    "modern country",
    "motown",
    "mumble rap",
    "musique concrète",
    "neo-soul",
    "neoclassical",
    "neurofunk",
    "new age",
    "new jack swing",
    "new traditionalist",
    "noise",
    "noise rock",
    "norteño",
    "northern soul",
    "nu jazz",
    "nu-disco",
    "nu-metal",
    "nwobhm (new wave of british heavy metal)",
    "ny drill",
    "old-time",
    "opera",
    "opera buffa",
    "operetta",
    "outlaw country",
    "p-funk",
    "pansori",
    "phonk",
    "piedmont blues",
    "pinoy rock",
    "pop",
    "pop country",
    "pop punk",
    "pop rock",
    "pop soul",
    "post-hardcore",
    "post-minimalism",
    "post-punk",
    "post-rock",
    "power metal",
    "power pop",
    "progressive folk rock",
    "progressive house",
    "progressive rock",
    "progressive rock (prog)",
    "proto-punk",
    "psychedelic",
    "psychedelic rock",
    "psytrance",
    "punk rock",
    "qawwali",
    "r&b",
    "ragtime",
    "rai",
    "ranchera",
    "rap",
    "reggae",
    "reggaeton",
    "renaissance",
    "requiem",
    "retrowave",
    "riddim",
    "riot grrrl",
    "rock",
    "rock and roll",
    "rock n roll",
    "rockabilly",
    "rocksteady",
    "romantic",
    "roots rock",
    "salsa",
    "samba",
    "screamo",
    "serialism (12-tone)",
    "shoegaze",
    "ska",
    "ska punk",
    "skiffle",
    "slap house",
    "sludge",
    "smooth jazz",
    "soca",
    "soft rock",
    "sophisti-pop",
    "soukous",
    "soul",
    "soul blues",
    "southern hip hop",
    "southern rock",
    "space rock",
    "stax",
    "stoner metal",
    "sufi music",
    "surf rock",
    "swamp blues",
    "swing",
    "symphonic metal",
    "synth-pop",
    "synthpop",
    "synthwave",
    "tango",
    "tarab",
    "tech house",
    "techno",
    "teen pop",
    "tejano",
    "texas blues",
    "thrash metal",
    "traditional chinese instrumental",
    "traditional country",
    "traditional folk",
    "traditional pop",
    "trance",
    "trap",
    "trip-hop",
    "tropical house",
    "trot",
    "uk drill",
    "uk garage",
    "uk hip hop",
    "uplifting trance",
    "vaporwave",
    "west coast hip hop",
    "west coast jazz",
    "western swing",
    "zouk"
}

# ============================================================
# 2. Alias de normalización
# ============================================================

GENRE_ALIASES = {
    "hip-hop": "hip hop",
    "hiphop": "hip hop",

    "rnb": "r&b",
    "r and b": "r&b",
    "rhythm and blues": "r&b",

    "rock & roll": "rock",
    "rock n' roll": "rock and roll",

    "edm": "electronic",
    "electronic dance music": "electronic",

    "contry": "country",
    "country-pop": "country pop",
    "alt country": "alt-country",
    "honky tonk": "honky-tonk",

    "folkrock": "folk rock",
    "blues-rock": "blues rock",
    "jazz rock": "jazz-rock",

    "singer-songwriter": "singer songwriter",
    "neo soul": "neo-soul",
    "synthpop": "synth-pop",
    "trip hop": "trip-hop"
}

# ============================================================
# 3. Agrupación de subgéneros en macrogéneros
# ============================================================

GENRE_GROUPS = {
    "Rock": [
        "acid rock", "alt rock", "alternative", "alternative rock", "art rock",
        "blues rock", "britpop", "country rock", "folk rock", "garage rock",
        "grunge", "hard rock", "heartland rock", "indie", "indie rock",
        "math rock", "noise rock", "pinoy rock", "pop rock", "post-rock",
        "progressive rock", "progressive rock (prog)", "psychedelic",
        "psychedelic rock", "riot grrrl", "rock", "rock n roll",
        "rockabilly", "roots rock", "shoegaze", "soft rock",
        "southern rock", "space rock", "surf rock"
    ],

    "Pop": [
        "afro-pop", "arabic pop", "art pop", "bubblegum pop", "c-pop",
        "chamber pop", "contemporary pop", "country pop", "dance-pop",
        "dream pop", "electro pop", "electropop", "folk pop", "hyperpop",
        "indie pop", "j-pop", "k-pop", "latin pop", "pop", "power pop",
        "sophisti-pop", "synth-pop", "teen pop", "traditional pop"
    ],

    "Hip Hop / Rap": [
        "alternative hip hop", "boom bap", "chicago drill", "cloud rap",
        "conscious hip hop", "country rap", "drill", "east coast hip hop",
        "emo rap", "g-funk", "gangsta rap", "grime", "hip hop", "jazz rap",
        "k-hip hop", "latin hip hop", "latin trap", "lo-fi hip hop",
        "midwest hip hop", "mumble rap", "ny drill", "phonk", "rap",
        "southern hip hop", "trap", "uk drill", "uk hip hop",
        "west coast hip hop"
    ],

    "Electronic / Dance": [
        "acid house", "acid techno", "amapiano", "ambient", "bass music",
        "bassline", "brostep", "chillout", "chillstep", "chiptune (8-bit)",
        "cyberpunk", "dance", "deep house", "detroit techno", "disco",
        "downtempo", "drum and bass", "dubstep", "electro house",
        "electro swing", "electroclash", "electronic",
        "electronic & dance (edm)", "french house", "frenchcore",
        "future bass", "gabber", "glitch", "goa trance", "hard dance",
        "hard trance", "hardstyle", "house", "idm (intelligent dance music)",
        "illbient", "indietronica", "industrial techno", "italo disco",
        "jungle", "liquid drum and bass", "minimal techno", "neurofunk",
        "nu-disco", "progressive house", "psytrance", "retrowave", "riddim",
        "slap house", "synthwave", "tech house", "techno", "trance",
        "trip-hop", "tropical house", "uk garage", "uplifting trance",
        "vaporwave"
    ],

    "Jazz": [
        "acid jazz", "afro-cuban jazz", "avant-garde jazz", "bebop",
        "big band", "cool jazz", "dixieland", "early jazz", "free jazz",
        "gypsy jazz", "hard bop", "jazz", "jazz funk", "jazz-rock",
        "latin jazz", "nu jazz", "ragtime", "smooth jazz", "swing",
        "west coast jazz"
    ],

    "Blues": [
        "blues", "chicago blues", "classic blues", "country blues",
        "delta blues", "jump blues", "memphis blues", "piedmont blues",
        "soul blues", "swamp blues", "texas blues"
    ],

    "Country": [
        "alt-country", "american country", "appalachian", "bluegrass",
        "bro-country", "canadian country", "classic country",
        "contemporary country", "country", "country and western",
        "country two step", "countrypolitan", "honky-tonk",
        "modern country", "outlaw country", "pop country",
        "traditional country", "western swing"
    ],

    "R&B / Soul / Funk": [
        "alternative r&b", "boogie", "classic r&b", "contemporary r&b",
        "doo-wop", "funk", "go-go", "gospel", "motown", "neo-soul",
        "new jack swing", "northern soul", "p-funk", "pop soul",
        "r&b", "soul", "stax"
    ],

    "Metal": [
        "alternative metal", "alternative/modern metal", "black metal",
        "classic metal", "death metal", "deathcore", "doom metal",
        "extreme metal", "glam metal", "gothic metal", "heavy metal",
        "melodic death metal", "metal", "metalcore", "nu-metal",
        "nwobhm (new wave of british heavy metal)", "power metal",
        "sludge", "stoner metal", "symphonic metal", "thrash metal"
    ],

    "Folk / Traditional": [
        "anti-folk", "celtic", "contemporary folk", "folk", "freak folk",
        "indie folk", "old-time", "progressive folk rock", "skiffle",
        "traditional folk"
    ],

    "Classical / Art Music": [
        "avant-garde", "baroque", "carnatic classical", "choral & liturgical",
        "classical", "early music", "expressionism", "grand opera",
        "gregorian chant", "hindustani classical", "impressionism", "mass",
        "medieval", "minimalism", "musique concrète", "neoclassical",
        "opera", "opera buffa", "operetta", "post-minimalism",
        "renaissance", "requiem", "romantic", "serialism (12-tone)",
        "traditional chinese instrumental"
    ],

    "Latin": [
        "bachata", "bossa nova", "cumbia", "mariachi", "merengue",
        "norteño", "ranchera", "reggaeton", "salsa", "samba",
        "tango", "tejano"
    ],

    "Reggae / Caribbean": [
        "calypso", "dancehall", "dub", "reggae", "rocksteady",
        "ska", "soca", "zouk"
    ],

    "Punk / Emo / Hardcore": [
        "emo", "emocore", "hardcore", "hardcore punk", "pop punk",
        "post-hardcore", "post-punk", "proto-punk", "punk rock",
        "screamo", "ska punk"
    ],

    "Global / World": [
        "afrobeats", "bhangra", "bollywood", "dangdut", "enka",
        "gagaku", "gamelan", "highlife", "jùjú", "klezmer",
        "kwaito", "mbalax", "pansori", "qawwali", "rai",
        "soukous", "sufi music", "tarab", "trot"
    ],

    "Experimental / Other": [
        "experimental", "fusion", "krautrock", "new age",
        "new traditionalist", "noise"
    ]
}

# ============================================================
# 3B. Añadir géneros reales detectados en el dataset raw
#     Pegar ANTES de crear GENRE_TO_GROUP
# ============================================================

# ------------------------------------------------------------
# Asegurar que los nombres finales coinciden con el objetivo
# ------------------------------------------------------------

# Tu objetivo usa "Experimental" y "Other", no "Experimental / Other" ni "Global / World"
old_global_world = GENRE_GROUPS.pop("Global / World", [])
old_experimental_other = GENRE_GROUPS.pop("Experimental / Other", [])

GENRE_GROUPS.setdefault("Experimental", [])
GENRE_GROUPS.setdefault("Other", [])


def add_to_group(group, genres):
    """
    Añade géneros a un macrogénero y también los incorpora a VALID_GENRES.
    Evita duplicados.
    """
    GENRE_GROUPS.setdefault(group, [])

    existing = set(GENRE_GROUPS[group])

    for genre in genres:
        genre = str(genre).strip().lower()

        if genre not in existing:
            GENRE_GROUPS[group].append(genre)
            existing.add(genre)

        VALID_GENRES.add(genre)


# ------------------------------------------------------------
# Reasignar antiguos grupos no objetivo
# ------------------------------------------------------------

for genre in old_global_world:
    add_to_group("Other", [genre])

for genre in old_experimental_other:
    if genre in {"experimental", "krautrock", "noise"}:
        add_to_group("Experimental", [genre])
    elif genre == "fusion":
        add_to_group("Jazz", [genre])
    elif genre == "new traditionalist":
        add_to_group("Country", [genre])
    else:
        add_to_group("Other", [genre])


# ------------------------------------------------------------
# Alias nuevos encontrados en el dataset raw
# ------------------------------------------------------------

GENRE_ALIASES.update({
    "-death metal-": "death metal",

    # Rock / variants
    "50s rock n roll": "rock and roll",
    "rock 'n' roll": "rock and roll",
    "rock & roll": "rock and roll",
    "rock n' roll": "rock and roll",
    "rock - classic": "classic rock",
    "rock - melodic": "rock",
    "rock ballads": "rock",
    "rockpop": "pop rock",
    "poprock": "pop rock",
    "blues-rock": "blues rock",
    "bluesrock": "blues rock",
    "folk-rock": "folk rock",
    "folkrock": "folk rock",
    "prog rock": "progressive rock",
    "80s hard rock": "hard rock",

    # Pop
    "bubblegum": "bubblegum pop",
    "dance pop": "dance-pop",
    "pop music tag": "pop",
    "synth pop": "synth-pop",
    "synthpop": "synth-pop",
    "powerpop": "power pop",

    # Country
    "alt country": "alt-country",
    "country-pop": "country pop",
    "honky tonk": "honky-tonk",
    "early 90s country": "country",
    "late 80s country": "country",
    "mid 90s country": "country",

    # Hip Hop / Rap
    "hip-hop": "hip hop",
    "hiphop": "hip hop",
    "alternative hip-hop": "alternative hip hop",
    "east coast rap": "east coast hip hop",
    "southern rap": "southern hip hop",
    "underground hip-hop": "underground hip hop",
    "underground rap": "underground hip hop",

    # R&B / Soul / Funk
    "rnb": "r&b",
    "rb": "r&b",
    "r and b": "r&b",
    "rhythm and blues": "r&b",
    "rhythm and blues tag": "r&b",
    "rhythum and blues tag": "r&b",
    "80s rhythm and blues": "r&b",
    "alternative rnb": "alternative r&b",
    "mid rnb": "r&b",
    "rnb christmas": "r&b",
    "rnbxmas": "r&b",
    "soul music": "soul",

    # Electronic / Dance
    "edm": "electronic",
    "electronic dance music": "electronic",
    "eletronic": "electronic",
    "electronica": "electronic",
    "electro": "electronic",
    "dance-house": "dance",
    "disco tag": "disco",
    "dnb": "drum and bass",
    "drum n bass": "drum and bass",
    "euro dance": "eurodance",
    "goa": "goa trance",
    "idm": "idm (intelligent dance music)",
    "festival progressive house": "progressive house",
    "psychedelic trance": "psytrance",
    "house tribal brasil": "house",

    # Metal
    "80s power metal": "power metal",
    "avant-gard metal": "avant-garde metal",
    "neo-classical metal": "neoclassical metal",
    "nu metal": "nu-metal",
    "nwobhm": "nwobhm (new wave of british heavy metal)",
    "prog metal": "progressive metal",
    "symphonic black": "symphonic black metal",

    # Folk / Traditional
    "folk - celtic": "celtic",
    "singer-songwriter": "singer songwriter",
    "singer/songwriter": "singer songwriter",
    "traditional": "traditional folk",

    # Classical / Art Music
    "classic music": "classical",

    # Jazz / Blues
    "jazz rock": "jazz-rock",
    "jazzy": "jazz",
    "bluesy": "blues",

    # Reggae / Caribbean
    "two-tone": "ska",

    # Other
    "a capella": "a cappella",
    "acostic guitar": "acoustic",
    "ost": "soundtrack",
    "score": "soundtrack",
    "christmas": "christmas music",
    "xmas": "christmas music",
    "yule": "christmas music",
    "christmas songs": "christmas music",
    "classic christmas": "christmas music",
    "praise & worship": "praise and worship"
})


# ------------------------------------------------------------
# Nuevos géneros reales por macrogénero
# ------------------------------------------------------------

add_to_group("Rock", [
    "acoustic rock",
    "aor",
    "arena rock",
    "cock rock",
    "electronic rock",
    "girl rock",
    "glam",
    "glam rock",
    "gothic rock",
    "industrial rock",
    "lounge rock",
    "new wave",
    "piano rock",
    "post-grunge",
    "slow rock",
    "stoner rock",
    "synth-rock",
    "trip rock",
    "whiskey-rock"
])

add_to_group("Pop", [
    "adult contemporary",
    "am pop",
    "baroque pop",
    "boyband",
    "boybands",
    "christian pop",
    "dark pop",
    "girl group",
    "girlsband",
    "jazz pop",
    "psychedelic pop"
])

add_to_group("Country", [
    "brazilian country",
    "countrycore",
    "sertanejo"
])

add_to_group("Electronic / Dance", [
    "alternative dance",
    "atlanta bass",
    "big beat",
    "breakbeat",
    "breakcore",
    "chill house",
    "classic freestyle",
    "dark ambient",
    "darkwave",
    "dreamwave",
    "dutch house",
    "electro hop",
    "eurodance",
    "freestyle",
    "happy hardcore",
    "hard techno",
    "hi-nrg",
    "hip house",
    "indie electronic",
    "italodance",
    "jumpstyle",
    "rave",
    "schranz",
    "vocal trance",
    "wonky"
])

add_to_group("Folk / Traditional", [
    "acoustic",
    "americana",
    "fado",
    "folk singer-songwriter",
    "irish folk",
    "new weird america",
    "singer songwriter"
])

add_to_group("R&B / Soul / Funk", [
    "chi-town soul",
    "classic soul",
    "contemporary gospel",
    "old school soul",
    "psychedelic soul",
    "quiet storm",
    "retro soul",
    "slowjamz",
    "southern gospel",
    "synth funk",
    "urban",
    "urban soul"
])

add_to_group("Punk / Emo / Hardcore", [
    "beatdown",
    "bubblegum punk",
    "celtic punk",
    "crunkcore",
    "dance-punk",
    "deathrock",
    "folk punk",
    "nintendocore",
    "punk",
    "riot grrrl",
    "skate punk",
    "straight edge",
    "trancecore"
])

add_to_group("Metal", [
    "ambient black metal",
    "atmospheric metal",
    "avant-garde metal",
    "black death metal",
    "blackened death metal",
    "brazilian thrash metal",
    "brutal death metal",
    "brutal deathcore",
    "celtic metal",
    "christian heavy metal",
    "christian metal",
    "dark metal",
    "death doom metal",
    "death n roll",
    "djent",
    "epic metal",
    "female fronted metal",
    "folk metal",
    "german thrash metal",
    "goregrind",
    "grindcore",
    "groove metal",
    "hair metal",
    "industrial black metal",
    "industrial metal",
    "italian metal",
    "love metal",
    "melodic black metal",
    "melodic metal",
    "metal ballad",
    "metal ballads",
    "neoclassical metal",
    "norwegian black metal",
    "old school thrash metal",
    "opera metal",
    "pagan metal",
    "party metal",
    "pirate metal",
    "progressive black metal",
    "progressive metal",
    "progressive power metal",
    "speed metal",
    "symphonic black metal",
    "symphonic gothic metal",
    "technical thrash metal",
    "vampiric metal",
    "viking metal"
])

add_to_group("Hip Hop / Rap", [
    "abstract hip hop",
    "alternative rap",
    "christian rap",
    "crunk",
    "dirty south",
    "instrumental hip hop",
    "rapcore",
    "underground hip hop"
])

add_to_group("Experimental", [
    "experimental rock",
    "industrial"
])

add_to_group("Jazz", [
    "jazz fusion",
    "jazz pop",
    "vocal jazz",
    "west coast swing"
])

add_to_group("Latin", [
    "axe",
    "latin",
    "latin freestyle",
    "mpb",
    "samba rock"
])

add_to_group("Reggae / Caribbean", [
    "bashment",
    "christian reggae"
])

add_to_group("Classical / Art Music", [
    "hymns",
    "waltz"
])

add_to_group("Other", [
    "a cappella",
    "ballad",
    "ballads",
    "christian",
    "christmas music",
    "comedy",
    "contemporary christian",
    "dark cabaret",
    "easy listening",
    "instrumental",
    "interlude",
    "lo-fi",
    "musical",
    "musicals",
    "poetry",
    "praise and worship",
    "soundtrack",
    "spoken word",
    "world",
    "worship"
])


# ------------------------------------------------------------
# Seguridad extra: cualquier género que esté en GENRE_GROUPS
# debe estar también en VALID_GENRES
# ------------------------------------------------------------

VALID_GENRES.update({
    genre
    for genres in GENRE_GROUPS.values()
    for genre in genres
})

VALID_GENRES.update(set(GENRE_ALIASES.values()))

# ============================================================
# 4. Crear diccionario inverso: subgénero -> macrogénero
# ============================================================

GENRE_TO_GROUP = {}

for group, genres in GENRE_GROUPS.items():
    for genre in genres:
        genre_norm = str(genre).strip().lower()
        genre_norm = GENRE_ALIASES.get(genre_norm, genre_norm)
        GENRE_TO_GROUP[genre_norm] = group

# ============================================================
# 5. Función para normalizar subgéneros
# ============================================================

def normalize_genre(genre):
    """
    Normaliza una etiqueta de género individual.
    Devuelve el subgénero normalizado si pertenece a VALID_GENRES.
    Si no pertenece, devuelve None.
    """

    if pd.isna(genre):
        return None

    genre = str(genre)
    genre = html.unescape(genre)
    genre = genre.strip().lower()

    # Quitar comillas externas
    genre = genre.strip("'").strip('"')

    # Normalizar espacios y separadores
    genre = genre.replace("_", " ")
    genre = re.sub(r"\s+", " ", genre).strip()

    # Quitar puntuación externa sin romper r&b, trip-hop o paréntesis internos
    genre = genre.strip(" .;:,")

    # Aplicar alias
    genre = GENRE_ALIASES.get(genre, genre)

    # Validar
    if genre not in VALID_GENRES:
        return None

    return genre

# ============================================================
# 6. Función para convertir subgénero a macrogénero
# ============================================================

def map_genre_to_group(genre):
    """
    Convierte un subgénero en macrogénero.
    """

    genre_norm = normalize_genre(genre)

    if genre_norm is None:
        return None

    return GENRE_TO_GROUP.get(genre_norm, "Other")

# ============================================================
# 7. Parsear celda y devolver macrogéneros
# ============================================================

def parse_genre_cell(value):
    """
    Convierte una celda de género en una lista de macrogéneros.

    Soporta:
    - valor simple: "pop"
    - lista en string: "['pop', 'rock']"
    - valores separados por comas: "pop, rock, r&b"

    Devuelve:
    - ["Pop"]
    - ["Rock"]
    - ["Hip Hop / Rap", "Pop"]
    """

    if pd.isna(value):
        return []

    value = str(value).strip()

    parsed_values = []

    # Caso lista en string: "['pop', 'rock']"
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (list, tuple, set)):
                parsed_values = list(parsed)
            else:
                parsed_values = [value]

        except Exception:
            parsed_values = [value]

    # Caso separado por comas: "pop, rock, r&b"
    elif "," in value:
        parsed_values = value.split(",")

    # Caso simple
    else:
        parsed_values = [value]

    genre_groups = []

    for genre in parsed_values:
        group = map_genre_to_group(genre)

        if group is not None:
            genre_groups.append(group)

    # Eliminar duplicados manteniendo el orden
    unique_groups = []
    seen = set()

    for group in genre_groups:
        if group not in seen:
            unique_groups.append(group)
            seen.add(group)

    return unique_groups

# ============================================================
# 8. Aplicar a genre_1, genre_2 y genre_3
# ============================================================

genre_cols = ["genre_1", "genre_2", "genre_3"]

for col in genre_cols:
    df[col + "_clean"] = df[col].apply(
        lambda x: parse_genre_cell(x)[0] if len(parse_genre_cell(x)) > 0 else None
    )

# ============================================================
# 9. Construir lista final de macrogéneros por canción
# ============================================================

def build_genre_list(row):
    """
    Construye una lista de macrogéneros únicos por canción.
    """

    genres = []

    for col in ["genre_1_clean", "genre_2_clean", "genre_3_clean"]:
        genre = row[col]

        if genre is not None and pd.notna(genre):
            genres.append(genre)

    unique_genres = []
    seen = set()

    for genre in genres:
        if genre not in seen:
            unique_genres.append(genre)
            seen.add(genre)

    return unique_genres


df["genre_list"] = df.apply(build_genre_list, axis=1)
df["n_genres"] = df["genre_list"].apply(len)

# ============================================================
# 10. Comprobaciones rápidas
# ============================================================

print("Ejemplo parse_genre_cell('alternative rock'):", parse_genre_cell("alternative rock"))
print("Ejemplo parse_genre_cell('pop, r&b, trap'):", parse_genre_cell("pop, r&b, trap"))

print("\nNúmero de canciones sin macrogénero válido:")
print((df["n_genres"] == 0).sum())
print("Número de canciones con al menos un macrogénero válido:")
print((df["n_genres"] > 0).sum())

print("\nDistribución de macrogéneros:")
display(
    df.explode("genre_list")["genre_list"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "genre_group", "genre_list": "n_songs"})
)

display(
    df[[
        "artist_clean",
        "song",
        "genre_1",
        "genre_2",
        "genre_3",
        "genre_1_clean",
        "genre_2_clean",
        "genre_3_clean",
        "genre_list",
        "n_genres"
    ]].tail(20)
)

Ejemplo parse_genre_cell('alternative rock'): ['Rock']
Ejemplo parse_genre_cell('pop, r&b, trap'): ['Pop', 'R&B / Soul / Funk', 'Hip Hop / Rap']

Número de canciones sin macrogénero válido:
4578
Número de canciones con al menos un macrogénero válido:
5503

Distribución de macrogéneros:


,n_songs,count
0,Rock,1349
1,Pop,1055
2,Metal,914
3,Hip Hop / Rap,871
4,Electronic / Dance,847
5,Country,805
6,Folk / Traditional,720
7,Other,690
8,R&B / Soul / Funk,579
9,Punk / Emo / Hardcore,222


,artist_clean,song,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean,genre_list,n_genres
10061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10064,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10066,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10067,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10068,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0
10070,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],0


In [82]:
# NORMALIZAR GENERO

genre_cols = ["genre_1", "genre_2", "genre_3"]

for col in genre_cols:
    df[col + "_clean"] = df[col].apply(
        lambda x: parse_genre_cell(x)[0] if len(parse_genre_cell(x)) > 0 else None
    )

display(df[["genres", "genre_1", "genre_2", "genre_3", 
            "genre_1_clean", "genre_2_clean", "genre_3_clean"]].head())

,genres,genre_1,genre_2,genre_3,genre_1_clean,genre_2_clean,genre_3_clean
0,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
1,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
2,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
3,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal
4,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,Metal,Metal,Metal


In [83]:
def count_words(text):
    if pd.isna(text):
        return 0
    return len(re.findall(r"\b[\w']+\b", str(text)))

df["word_count"] = df["lyrics"].apply(count_words)

MIN_WORDS = 1

initial_rows = len(df)

mask_valid_lyrics = df["lyrics"].notna() & (df["word_count"] >= MIN_WORDS)
mask_valid_genres = df["n_genres"] > 0

df_clean = df[mask_valid_lyrics & mask_valid_genres].copy()

print("Filas iniciales:", initial_rows)
print("Filas tras limpieza:", len(df_clean))
print("Canciones eliminadas:", initial_rows - len(df_clean))

print("\nMotivos principales:")
print("Sin letra válida:", (~mask_valid_lyrics).sum())
print("Sin género válido:", (~mask_valid_genres).sum())

Filas iniciales: 10081
Filas tras limpieza: 4673
Canciones eliminadas: 5408

Motivos principales:
Sin letra válida: 4814
Sin género válido: 4578


In [84]:
def normalize_text_id(text):
    """
    Normaliza texto para detectar duplicados.
    """
    
    if pd.isna(text):
        return ""
    
    text = str(text).strip().lower()
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text)
    
    return text


df_clean["artist_norm"] = df_clean["artist"].apply(normalize_text_id)
df_clean["song_norm"] = df_clean["song"].apply(normalize_text_id)

duplicates = df_clean.duplicated(subset=["artist_norm", "song_norm"], keep=False)

print("duplicados:", duplicates.sum())

df_clean = (
    df_clean
    .sort_values("word_count", ascending=False)
    .drop_duplicates(subset=["artist_norm", "song_norm"], keep="first")
    .sort_index()
    .copy()
)

print("Shape tras eliminar duplicados:", df_clean.shape)

duplicados: 108
Shape tras eliminar duplicados: (4619, 16)


In [85]:
# Dataset expandido por genero

df_genres = df_clean.explode("genre_list").copy()

df_genres = df_genres.rename(columns={"genre_list": "genre"})

df_genres = df_genres[df_genres["genre"].notna()].copy()

# Evitar que una misma canción aparezca dos veces en el mismo género
df_genres = df_genres.drop_duplicates(
    subset=["artist_norm", "song_norm", "genre"],
    keep="first"
)

print("Shape df_clean, una fila por canción:", df_clean.shape)
print("Shape df_genres, una fila por canción-género:", df_genres.shape)

display(df_genres[["artist_clean", "song", "genre", "lyrics", "word_count"]].head())

Shape df_clean, una fila por canción: (4619, 16)
Shape df_genres, una fila por canción-género: (6892, 16)


,artist_clean,song,genre,lyrics,word_count
0,moonspell,Heartshaped Abyss,Metal,Never resist\r\nThe firewalker\r\nHe has been ...,167
1,moonspell,Let The Children Cum To Me...,Metal,"Hey you little Jesus' bride, why have you smil...",197
2,moonspell,Memento Mori,Metal,"I am the moment, the soul\r\nThe moment that e...",219
3,moonspell,Once It Was Ours!,Metal,Vanishing act inside the weak\r\nIn need of yo...,174
4,moonspell,Serpent Angel,Metal,Father Satan send the Serpent\r\nPoison me wit...,152


In [86]:
genre_counts = df_genres["genre"].value_counts()


# 1. Le decimos a Pandas que no ponga límite máximo de filas
#pd.set_option('display.max_rows', None)

# 2. Mostramos tu tabla (sin el .head())
display(
    genre_counts
    .reset_index()
    .rename(columns={"index": "genre", "genre": "n_songs"})
)

# 3. Restauramos el límite normal (por defecto suele ser 60)
# Esto es importante para que futuras tablas grandes no te bloqueen la pantalla
pd.reset_option('display.max_rows')

print("Número total de géneros únicos:", df_genres["genre"].nunique())
print("Número total de registros canción-género:", len(df_genres))


,n_songs,count
0,Rock,1196
1,Metal,887
2,Hip Hop / Rap,790
3,Pop,781
4,Electronic / Dance,630
5,Other,581
6,Folk / Traditional,572
7,Country,548
8,R&B / Soul / Funk,487
9,Punk / Emo / Hardcore,213


Número total de géneros únicos: 16
Número total de registros canción-género: 6892


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

#df_clean.to_csv(OUTPUT_DIR / "songs_clean_by_song.csv", index=False)
df_genres.to_csv(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv", index=False)

print("Archivos guardados:")
print(OUTPUT_DIR / "songs_clean_by_genre_exploded.csv")


Archivos guardados:
outputs/songs_clean_by_genre_exploded.csv


En primer lugar, se realizó una limpieza del dataset original con el objetivo de preparar los nombres de artistas y los géneros musicales para el análisis posterior. Se eliminaron canciones sin letra disponible, canciones con letras demasiado cortas y registros sin ningún género válido.

Además, se normalizaron las etiquetas de género presentes en las columnas genre_1, genre_2 y genre_3, convirtiéndolas a minúsculas, eliminando espacios innecesarios y unificando algunas variantes frecuentes. Dado que una canción puede pertenecer a varios géneros, se generó un segundo dataset en formato expandido, donde cada fila representa una combinación canción-género. Este formato permite analizar posteriormente la distribución emocional de las letras para cada género musical.

| Archivo                                      | Nivel                       | Uso principal                   |
| -------------------------------------------- | --------------------------- | ------------------------------- |
| `dataset_fusionado.csv`                    | Una fila por canción        | Visualización compacta     |
| `songs_clean_by_genre_exploded.csv`          | Una fila por canción-género | Análisis completo por género    |

